# XGBoost를 활용한 암종 분류

암환자의 4,384개 유전자 변이 정보를 이용해 26개 `SUBCLASS`를 분류하기 위한 초기 Notebook 골격입니다.

> 아직 모델 실험을 실행하지 않았습니다. 학습을 시작하기 전에 GitHub Issue와 EXP-ID를 예약하고, 실행 가능한 코드는 `scripts/`와 `configs/`로 이전해야 합니다.

## 0. 프로젝트 규칙 확인

- `PROJECT_CONTEXT.md`와 `EXPERIMENT_HISTORY.md`를 먼저 읽습니다.
- 공식 평가지표는 Macro F1입니다.
- 공용 split은 `data/splits/stratified_5fold_seed42.csv`입니다.
- 원본 데이터, 모델, OOF와 테스트 확률은 Git에 커밋하지 않습니다.
- Notebook 결과를 비교 실험으로 채택할 때는 config, script, metrics와 재현성 manifest를 함께 생성합니다.

In [ ]:
from pathlib import Path

import pandas as pd

from open_cancer.constants import CLASS_LABELS
from open_cancer.validation import validate_competition_data

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
FOLD_PATH = PROJECT_ROOT / "data" / "splits" / "stratified_5fold_seed42.csv"

CLASS_LABELS

## 1. 원본 데이터 계약 검증

학습 전에 파일 해시, 행·열 수, 클래스, ID와 유전자 컬럼 순서를 검증합니다.

In [ ]:
data_summary = validate_competition_data(
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
)
data_summary

## 2. 데이터와 공용 fold 연결

이 셀은 데이터를 읽고 fold 배정이 모든 학습 ID에 정확히 한 번 연결되는지만 확인합니다. 모델 학습은 수행하지 않습니다.

In [ ]:
train = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
test = pd.read_csv(TEST_PATH, dtype=str, keep_default_na=False)
folds = pd.read_csv(FOLD_PATH, dtype={"ID": str, "fold": int})

train = train.merge(folds, on="ID", how="left", validate="one_to_one")
assert train["fold"].notna().all()
assert set(train["fold"]) == {0, 1, 2, 3, 4}

gene_columns = [column for column in test.columns if column != "ID"]
assert gene_columns == [
    column for column in train.columns if column not in {"ID", "SUBCLASS", "fold"}
]

{
    "train_shape": train.shape,
    "test_shape": test.shape,
    "gene_columns": len(gene_columns),
    "fold_counts": train["fold"].value_counts().sort_index().to_dict(),
    "class_counts": train["SUBCLASS"].value_counts().sort_index().to_dict(),
}

## 3. EXP-001 시작 전 설계 항목

실제 XGBoost baseline을 시작할 때 다음 항목을 먼저 확정합니다.

1. GitHub 실험 Issue와 `EXP-001` 예약
2. `WT`, 빈 값, 단일·복수 변이에 대한 인코딩 방식
3. `configs/exp001_<slug>.yaml`의 전체 XGBoost 파라미터
4. fold별 seed, thread 수와 early stopping 규칙
5. OOF·테스트 확률·checkpoint·metrics 저장 경로
6. 제출 전 `INFERENCE_VERIFIED` 재현성 검증

학습 코드는 위 항목을 확정한 뒤 `scripts/run_exp001_<slug>.py`에 구현합니다.